# Old Photo Restoration Pipeline
### HumanFace8000 → Synthetic Degradation → CodeFormer + Real-ESRGAN → Evaluation

**Hardware target:** NVIDIA 3050 Ti (4GB VRAM), 16GB RAM  
**Dataset:** [HumanFace8000 on Kaggle](https://www.kaggle.com/datasets/)

## 1. Install Dependencies

In [ ]:
# Install core Python dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python pillow numpy matplotlib scikit-image lpips basicsr facexlib gfpgan

In [ ]:
# Clone CodeFormer and Real-ESRGAN repos
import os

if not os.path.exists('CodeFormer'):
    !git clone https://github.com/sczhou/CodeFormer.git

if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git

# Install CodeFormer requirements
!pip install -r CodeFormer/requirements.txt

# Install Real-ESRGAN
%cd Real-ESRGAN
!pip install basicsr
!pip install -r requirements.txt
!python setup.py develop
%cd ..

In [ ]:
# Download pretrained weights for CodeFormer
import os

os.makedirs('CodeFormer/weights/CodeFormer', exist_ok=True)
os.makedirs('CodeFormer/weights/facelib', exist_ok=True)

# CodeFormer model
!wget -q https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth \
    -O CodeFormer/weights/CodeFormer/codeformer.pth

# Face detection model
!wget -q https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/detection_Resnet50_Final.pth \
    -O CodeFormer/weights/facelib/detection_Resnet50_Final.pth

# Face parsing model
!wget -q https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/parsing_parsenet.pth \
    -O CodeFormer/weights/facelib/parsing_parsenet.pth

# Real-ESRGAN weights (for background enhancement)
os.makedirs('Real-ESRGAN/weights', exist_ok=True)
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth \
    -O Real-ESRGAN/weights/RealESRGAN_x2plus.pth

print('All weights downloaded!')

## 2. Load Dataset

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# ----------------------------------------------------------------
# UPDATE THIS PATH to where you extracted HumanFace8000 on your machine
# Example: DATASET_PATH = '/home/user/datasets/HumanFace8000/images'
# ----------------------------------------------------------------
DATASET_PATH = './HumanFace8000'  # <-- change this

# Output directories
os.makedirs('./output/original', exist_ok=True)
os.makedirs('./output/degraded', exist_ok=True)
os.makedirs('./output/restored', exist_ok=True)

# Collect image paths
supported_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_paths = [
    str(p) for p in Path(DATASET_PATH).rglob('*')
    if p.suffix.lower() in supported_exts
]

print(f'Found {len(image_paths)} images in dataset')

# Preview a few
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    img = Image.open(image_paths[i]).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'Sample {i+1}')
    ax.axis('off')
plt.suptitle('HumanFace8000 — Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Synthetic Degradation Pipeline

Each image gets a **randomized** combination of:
- Gaussian noise
- Gaussian blur
- JPEG compression artifacts
- Color fading / desaturation
- Random scratches

In [ ]:
import random
import io

def add_gaussian_noise(img: np.ndarray, std_range=(10, 50)) -> np.ndarray:
    """Add random Gaussian noise."""
    std = random.uniform(*std_range)
    noise = np.random.normal(0, std, img.shape).astype(np.float32)
    noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return noisy


def add_gaussian_blur(img: np.ndarray, kernel_range=(3, 9)) -> np.ndarray:
    """Apply random Gaussian blur."""
    k = random.choice(range(kernel_range[0], kernel_range[1] + 1, 2))  # odd kernels only
    sigma = random.uniform(0.5, 2.5)
    return cv2.GaussianBlur(img, (k, k), sigma)


def add_jpeg_artifacts(img: np.ndarray, quality_range=(10, 50)) -> np.ndarray:
    """Simulate JPEG compression artifacts."""
    quality = random.randint(*quality_range)
    pil_img = Image.fromarray(img)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf))


def add_color_fading(img: np.ndarray, fade_range=(0.3, 0.7)) -> np.ndarray:
    """Desaturate and fade colors to simulate old photo look."""
    fade_strength = random.uniform(*fade_range)
    # Convert to grayscale and blend back
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    faded = cv2.addWeighted(img, 1 - fade_strength, gray_rgb, fade_strength, 0)
    # Add slight yellowish tint (old paper effect)
    tint = np.array([255, 240, 200], dtype=np.float32)
    tint_strength = random.uniform(0.05, 0.15)
    tinted = cv2.addWeighted(faded.astype(np.float32), 1 - tint_strength,
                             np.ones_like(faded, dtype=np.float32) * tint, tint_strength, 0)
    return np.clip(tinted, 0, 255).astype(np.uint8)


def add_scratches(img: np.ndarray, num_scratches_range=(2, 8)) -> np.ndarray:
    """Draw random scratch lines to simulate physical damage."""
    result = img.copy()
    h, w = img.shape[:2]
    num_scratches = random.randint(*num_scratches_range)
    for _ in range(num_scratches):
        # Random scratch line
        x1, y1 = random.randint(0, w), random.randint(0, h)
        x2, y2 = random.randint(0, w), random.randint(0, h)
        color = random.choice([
            (255, 255, 255),  # white scratch
            (0, 0, 0),        # dark scratch
            (180, 160, 120),  # brown aged scratch
        ])
        thickness = random.randint(1, 2)
        cv2.line(result, (x1, y1), (x2, y2), color, thickness)
    return result


def degrade_image(img: np.ndarray, seed: int = None) -> np.ndarray:
    """
    Apply a randomized degradation pipeline to simulate an old photo.
    Set seed for reproducibility.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    degraded = img.copy()

    # Each degradation applied with some probability
    if random.random() > 0.2:  # 80% chance
        degraded = add_color_fading(degraded)

    if random.random() > 0.3:  # 70% chance
        degraded = add_gaussian_blur(degraded)

    if random.random() > 0.2:  # 80% chance
        degraded = add_gaussian_noise(degraded)

    if random.random() > 0.3:  # 70% chance
        degraded = add_jpeg_artifacts(degraded)

    if random.random() > 0.4:  # 60% chance
        degraded = add_scratches(degraded)

    return degraded


print('Degradation functions defined!')

In [ ]:
# Preview degradation on a few samples
NUM_PREVIEW = 4
fig, axes = plt.subplots(NUM_PREVIEW, 2, figsize=(10, 4 * NUM_PREVIEW))

for i in range(NUM_PREVIEW):
    original = np.array(Image.open(image_paths[i]).convert('RGB'))
    degraded = degrade_image(original, seed=i)  # seed per image for reproducibility

    axes[i, 0].imshow(original)
    axes[i, 0].set_title('Original')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(degraded)
    axes[i, 1].set_title('Degraded (Synthetic Old Photo)')
    axes[i, 1].axis('off')

plt.suptitle('Degradation Preview', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Generate and save degraded images for the full dataset (or a subset)
import shutil
from tqdm import tqdm

# Set how many images to process (use None for full dataset)
MAX_IMAGES = 100  # start small to test; set to None for all

paths_to_process = image_paths[:MAX_IMAGES] if MAX_IMAGES else image_paths

for idx, img_path in enumerate(tqdm(paths_to_process, desc='Degrading images')):
    fname = os.path.basename(img_path)
    stem = Path(img_path).stem

    # Save original copy
    shutil.copy(img_path, f'./output/original/{fname}')

    # Load, degrade, save
    original = np.array(Image.open(img_path).convert('RGB'))
    degraded = degrade_image(original, seed=idx)
    Image.fromarray(degraded).save(f'./output/degraded/{stem}.png')

print(f'Done! Degraded {len(paths_to_process)} images saved to ./output/degraded/')

## 4. Restoration — CodeFormer + Real-ESRGAN

We call CodeFormer's inference script as a subprocess.  
- `--fidelity_weight` controls the trade-off between quality and fidelity (0 = max enhancement, 1 = max fidelity to input)
- We use `0.5` as a balanced default — you can tune this.

In [ ]:
import subprocess

DEGRADED_DIR = './output/degraded'
RESTORED_DIR = './output/restored'
os.makedirs(RESTORED_DIR, exist_ok=True)

# CodeFormer inference
# --bg_upsampler realesrgan : uses Real-ESRGAN for background regions
# --face_upsample            : also upsamples faces via Real-ESRGAN after CodeFormer
# --fidelity_weight 0.5      : balance between restoration strength and input fidelity
# --upscale 2                : 2x upscaling

cmd = [
    'python', 'CodeFormer/inference_codeformer.py',
    '--input_path', DEGRADED_DIR,
    '--output_path', RESTORED_DIR,
    '--fidelity_weight', '0.5',
    '--upscale', '2',
    '--bg_upsampler', 'realesrgan',
    '--face_upsample',
    '--detection_model', 'retinaface_resnet50',
]

print('Running CodeFormer restoration...')
print(' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('Restoration complete!')

## 5. Evaluation — PSNR, SSIM, LPIPS

In [ ]:
import torch
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Load LPIPS model (AlexNet backbone, fast)
lpips_fn = lpips.LPIPS(net='alex')
lpips_fn.eval()


def compute_metrics(original: np.ndarray, restored: np.ndarray):
    """
    Compute PSNR, SSIM, and LPIPS between original and restored images.
    Images should be RGB numpy arrays (H, W, 3) in range [0, 255].
    """
    # Resize restored to match original if needed
    if original.shape != restored.shape:
        restored = cv2.resize(restored, (original.shape[1], original.shape[0]))

    # PSNR (higher is better)
    psnr_score = psnr(original, restored, data_range=255)

    # SSIM (higher is better)
    ssim_score = ssim(original, restored, channel_axis=2, data_range=255)

    # LPIPS (lower is better)
    def to_tensor(img):
        t = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0) / 127.5 - 1
        return t

    with torch.no_grad():
        lpips_score = lpips_fn(to_tensor(original), to_tensor(restored)).item()

    return {'PSNR': psnr_score, 'SSIM': ssim_score, 'LPIPS': lpips_score}


print('Evaluation functions ready!')

In [ ]:
from glob import glob

# CodeFormer saves results inside a 'final_results' subfolder
restored_files = glob(os.path.join(RESTORED_DIR, 'final_results', '*.png'))

if not restored_files:
    # Try root of restored dir
    restored_files = glob(os.path.join(RESTORED_DIR, '*.png'))

print(f'Found {len(restored_files)} restored images')

all_metrics = []

for restored_path in tqdm(restored_files, desc='Evaluating'):
    fname = os.path.basename(restored_path)
    original_path = os.path.join('./output/original', fname)

    # Try jpg version if png not found
    if not os.path.exists(original_path):
        original_path = original_path.replace('.png', '.jpg')
    if not os.path.exists(original_path):
        continue

    original = np.array(Image.open(original_path).convert('RGB'))
    restored = np.array(Image.open(restored_path).convert('RGB'))

    metrics = compute_metrics(original, restored)
    metrics['file'] = fname
    all_metrics.append(metrics)

# Summary
if all_metrics:
    avg_psnr  = np.mean([m['PSNR']  for m in all_metrics])
    avg_ssim  = np.mean([m['SSIM']  for m in all_metrics])
    avg_lpips = np.mean([m['LPIPS'] for m in all_metrics])

    print('\n===== Evaluation Summary =====')
    print(f'Images evaluated : {len(all_metrics)}')
    print(f'Avg PSNR         : {avg_psnr:.2f} dB  (higher is better)')
    print(f'Avg SSIM         : {avg_ssim:.4f}    (higher is better, max=1)')
    print(f'Avg LPIPS        : {avg_lpips:.4f}   (lower is better)')
else:
    print('No matched pairs found — check your file naming.')

## 6. Visual Comparison — Original vs Degraded vs Restored

In [ ]:
NUM_SHOW = min(5, len(restored_files))

fig, axes = plt.subplots(NUM_SHOW, 3, figsize=(15, 5 * NUM_SHOW))

for i, restored_path in enumerate(restored_files[:NUM_SHOW]):
    fname = os.path.basename(restored_path)
    stem = Path(fname).stem

    original_path = os.path.join('./output/original', fname)
    if not os.path.exists(original_path):
        original_path = original_path.replace('.png', '.jpg')

    degraded_path = os.path.join('./output/degraded', f'{stem}.png')

    original = Image.open(original_path).convert('RGB') if os.path.exists(original_path) else None
    degraded = Image.open(degraded_path).convert('RGB') if os.path.exists(degraded_path) else None
    restored = Image.open(restored_path).convert('RGB')

    if original:
        axes[i, 0].imshow(original)
    axes[i, 0].set_title('Original', fontsize=12)
    axes[i, 0].axis('off')

    if degraded:
        axes[i, 1].imshow(degraded)
    axes[i, 1].set_title('Degraded (Simulated Old Photo)', fontsize=12)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(restored)
    axes[i, 2].set_title('Restored (CodeFormer + Real-ESRGAN)', fontsize=12)
    axes[i, 2].axis('off')

plt.suptitle('Original vs Degraded vs Restored', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('./output/comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison saved to ./output/comparison.png')

## 7. (Optional) Tune Fidelity Weight

The `--fidelity_weight` parameter in CodeFormer controls the balance between:
- `0.0` → Maximum enhancement (may hallucinate details)
- `1.0` → Maximum fidelity to input (less enhancement)

Run the cell below to compare different fidelity weights on a single image.

In [ ]:
# Compare fidelity weights on a single image
TEST_IMAGE = './output/degraded/' + os.path.basename(restored_files[0])  # pick first degraded
FIDELITY_WEIGHTS = [0.0, 0.3, 0.5, 0.7, 1.0]

results = {}

for w in FIDELITY_WEIGHTS:
    out_dir = f'./output/fidelity_test/w{str(w).replace(".", "_")}'
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        'python', 'CodeFormer/inference_codeformer.py',
        '--input_path', TEST_IMAGE,
        '--output_path', out_dir,
        '--fidelity_weight', str(w),
        '--upscale', '2',
        '--bg_upsampler', 'realesrgan',
        '--face_upsample',
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    output_files = glob(os.path.join(out_dir, '**', '*.png'), recursive=True)
    if output_files:
        results[w] = output_files[0]
    print(f'w={w} done')

# Display comparison
n = len(results)
fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 5))

axes[0].imshow(Image.open(TEST_IMAGE).convert('RGB'))
axes[0].set_title('Degraded Input')
axes[0].axis('off')

for j, (w, path) in enumerate(results.items()):
    axes[j + 1].imshow(Image.open(path).convert('RGB'))
    axes[j + 1].set_title(f'fidelity={w}')
    axes[j + 1].axis('off')

plt.suptitle('Fidelity Weight Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('./output/fidelity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fidelity comparison saved!')